# Task C: Improved Seq2Seq Survival-Curve Pipeline

This notebook upgrades the first `task_c` baseline in three ways:

1. A stronger monotone decoder with explicit step embeddings.
2. Patience-based early stopping plus learning-rate scheduling.
3. A mean-curve baseline so the learned model is judged against something concrete.

The default dataset is now `dataset_v2` / `sampling_1`. The HDF5 files contain 2001 snapshots per trajectory, but this notebook intentionally uses the first `RAW_T_IN = 200` snapshots as the input prefix to predict future unbound outcomes.


In [ ]:
import copy
import os
import random
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

SEED = 42
GROUP_SIZE = 8
SURVIVAL_TRIALS = GROUP_SIZE
LOCAL_SMOKE_TEST = True
NUM_RESULT_BATCHES = 5  # Local subset. Use 50 on the server when all dataset_v2 shards are available.
RAW_T_IN = 200  # Use the first 200 HDF5 snapshots as the observed prefix.
TIME_STRIDE = 1  # Keep every snapshot in that prefix; do not confuse this with time_unbound units.
T_IN = len(range(0, RAW_T_IN, TIME_STRIDE))
# Survival target checkpoints in completed_systems_sampling_1.txt time_unbound units, not HDF5 array indices.
# dataset_v2 has 2001 snapshots, while time_unbound ranges up to 10000.
# The first 200 snapshots end at about time_unbound=1000, so train only on later horizons.
FUTURE_TIMES = np.arange(1100, 10000 + 1, 100, dtype=np.float32)
NUM_FEATURES = 15
SMOKE_MAX_CASES = 1000
MAX_CASES = SMOKE_MAX_CASES if LOCAL_SMOKE_TEST else None

BATCH_SIZE = 128
HIDDEN_DIM = 96
ENCODER_TYPE = "gru"  # one of: "gru", "cnn", "tcn"
GROUP_REDUCE = "mean"  # "mean" or "first"
DROPOUT = 0.20
EPOCHS = 3 if LOCAL_SMOKE_TEST else 80
LR = 1e-3
WEIGHT_DECAY = 2e-4
GRAD_CLIP_NORM = 1.0
TEST_RATIO = 0.2
VAL_RATIO = 0.1  # interpreted as a fraction of total cases, not of the remaining train pool
EARLY_STOPPING_PATIENCE = 12
EARLY_STOPPING_MIN_DELTA = 1e-4
SCHEDULER_PATIENCE = 4
SCHEDULER_FACTOR = 0.5
TEACHER_FORCING_RATIO = 0.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_DIR = Path("./dataset_v2")
SAMPLING_ID = 1
RESULT_PATHS = [
    DATASET_DIR / f"results_batch_{batch_id:02d}_sampling_{SAMPLING_ID}.hdf5"
    for batch_id in range(NUM_RESULT_BATCHES)
]
LABELS_PATH = DATASET_DIR / f"completed_systems_sampling_{SAMPLING_ID}.txt"

AVAILABLE_BATCH_IDS = list(range(NUM_RESULT_BATCHES))

print("Dataset dir:", DATASET_DIR)
print("Local smoke test:", LOCAL_SMOKE_TEST)
print("Result batches:", AVAILABLE_BATCH_IDS)
print("Max cases:", MAX_CASES)
print("Raw/effective input steps:", {"raw": RAW_T_IN, "stride": TIME_STRIDE, "effective": T_IN})
print("Future survival times in time_unbound units:", FUTURE_TIMES.tolist())


In [ ]:
RESULT_KEYS = [
    "aratios_complete",
    "mratios_complete",
    "eccs_complete",
    "incs_complete",
    "APs_complete",
    "LANs_complete",
]


def load_one_h5(path, keys, T=None, stride=1, max_rows=None, dtype=np.float32):
    time_sl = slice(None, None, stride) if T is None else slice(0, T, stride)
    row_sl = slice(None) if max_rows is None else slice(0, max_rows)
    out = {}
    with h5py.File(path, "r") as f:
        for key in keys:
            out[key] = f[key][row_sl, time_sl, ...].astype(dtype)
    return out


def load_all_batches(paths, keys, T=None, stride=1, max_rows=None, dtype=np.float32):
    shards = []
    remaining = None if max_rows is None else int(max_rows)
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(path)
        if remaining is not None and remaining <= 0:
            break
        shard = load_one_h5(path, keys=keys, T=T, stride=stride, max_rows=remaining, dtype=dtype)
        shards.append(shard)
        if remaining is not None:
            remaining -= shard[keys[0]].shape[0]

    if not shards:
        raise ValueError("No HDF5 shards were loaded")

    return {key: np.concatenate([shard[key] for shard in shards], axis=0) for key in keys}


def sincos(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)


def build_orbital_feature_stack(data: dict) -> np.ndarray:
    ar = data["aratios_complete"].astype(np.float32)
    mr = data["mratios_complete"].astype(np.float32)
    ec = data["eccs_complete"].astype(np.float32)
    inc = data["incs_complete"].astype(np.float32)
    ap = sincos(data["APs_complete"])
    lan = sincos(data["LANs_complete"])
    X = np.concatenate([ar, mr, ec, inc, ap, lan], axis=-1)
    assert X.shape[-1] == NUM_FEATURES, X.shape
    return X.astype(np.float32)


max_trajectories = None if MAX_CASES is None else MAX_CASES * GROUP_SIZE
data = load_all_batches(
    RESULT_PATHS,
    keys=RESULT_KEYS,
    T=RAW_T_IN,
    stride=TIME_STRIDE,
    max_rows=max_trajectories,
)
X_traj = build_orbital_feature_stack(data)
assert X_traj.shape[1] == T_IN, (X_traj.shape, T_IN)
print("Trajectory feature tensor:", X_traj.shape)


In [ ]:
labels = pd.read_csv(
    LABELS_PATH,
    sep=r"\s+",
    header=None,
    names=["index", "num_batch", "time_unbound", "ignore_other_than_minus1", "diagnosis"],
)

labels = labels.astype(
    {
        "index": "int64",
        "num_batch": "int64",
        "time_unbound": "float32",
        "ignore_other_than_minus1": "float32",
        "diagnosis": "int64",
    }
)

# Match the concatenated result-file order and ignore labels for result shards that are not present locally.
labels = labels[labels["num_batch"].isin(AVAILABLE_BATCH_IDS)].copy()
labels = labels.sort_values(["num_batch", "index"], ascending=True).reset_index(drop=True)
if max_trajectories is not None:
    labels = labels.iloc[:max_trajectories].reset_index(drop=True)
labels["raw_case_id"] = np.arange(len(labels)) // GROUP_SIZE

assert len(labels) == len(X_traj), (len(labels), len(X_traj))

valid_label_mask = (labels["time_unbound"] >= 0) & (labels["diagnosis"] == 0)
invalid_label_count = int((~valid_label_mask).sum())
if invalid_label_count:
    print(
        f"Dropping {invalid_label_count} trajectories with invalid survival labels "
        "(time_unbound < 0 or diagnosis != 0)."
    )
    X_traj = X_traj[valid_label_mask.to_numpy()]
    labels = labels.loc[valid_label_mask].reset_index(drop=True)

# Keep finite trajectory prefixes only.
finite_mask = np.isfinite(X_traj).all(axis=(1, 2))
X_traj = X_traj[finite_mask]
labels = labels.loc[finite_mask].reset_index(drop=True)

# Cleaning can break a perturbation set, so keep complete 8-sample cases only.
case_counts = labels["raw_case_id"].value_counts().sort_index()
complete_case_ids = case_counts[case_counts == GROUP_SIZE].index.to_numpy()
complete_mask = labels["raw_case_id"].isin(complete_case_ids).to_numpy()

X_traj = X_traj[complete_mask]
labels = labels.loc[complete_mask].reset_index(drop=True)
labels["case_id"], _ = pd.factorize(labels["raw_case_id"], sort=True)


def build_case_inputs(X: np.ndarray, case_ids: np.ndarray, reduce: str = "mean") -> np.ndarray:
    xs = []
    for case_id in np.unique(case_ids):
        group = X[case_ids == case_id]
        if reduce == "mean":
            xs.append(group.mean(axis=0))
        elif reduce == "first":
            xs.append(group[0])
        else:
            raise ValueError(f"Unsupported reduce mode: {reduce}")
    return np.stack(xs).astype(np.float32)


def build_survival_targets(time_unbound: np.ndarray, case_ids: np.ndarray, future_times: np.ndarray) -> np.ndarray:
    ys = []
    for case_id in np.unique(case_ids):
        group_times = time_unbound[case_ids == case_id].astype(np.float32)
        # Soft label y=N/8. The binomial loss below interprets this as N surviving trajectories out of 8.
        curve = np.array([(group_times >= t).mean() for t in future_times], dtype=np.float32)
        ys.append(curve)
    y = np.stack(ys)
    assert np.all(y[:, 1:] <= y[:, :-1] + 1e-6)
    return y


case_ids = labels["case_id"].to_numpy()
time_unbound = labels["time_unbound"].to_numpy()

X_case = build_case_inputs(X_traj, case_ids=case_ids, reduce=GROUP_REDUCE)
y_case = build_survival_targets(time_unbound, case_ids=case_ids, future_times=FUTURE_TIMES)

print("After finite filtering and complete-case filtering:")
print("  trajectories:", X_traj.shape[0])
print("  cases      :", y_case.shape[0])
print("  case input :", X_case.shape)
print("  case target:", y_case.shape)
print("  target tail distribution:", pd.Series(y_case[:, -1]).describe())
print("  target tail count distribution:", pd.Series(np.rint(y_case[:, -1] * SURVIVAL_TRIALS).astype(int)).describe())


In [ ]:
def _allocate_stratified_counts(bin_sizes: np.ndarray, target_total: int) -> np.ndarray:
    if target_total < 0 or target_total > int(bin_sizes.sum()):
        raise ValueError(f"Invalid target_total={target_total} for bin_sizes={bin_sizes}")

    raw = bin_sizes.astype(np.float64)
    if raw.sum() == 0:
        return np.zeros_like(bin_sizes, dtype=np.int64)

    ideal = raw / raw.sum() * target_total
    base = np.floor(ideal).astype(np.int64)
    remainder = int(target_total - base.sum())

    if remainder > 0:
        order = np.argsort(-(ideal - base))
        for i in order[:remainder]:
            base[i] += 1

    base = np.minimum(base, bin_sizes.astype(np.int64))
    deficit = int(target_total - base.sum())
    if deficit > 0:
        spare_order = np.argsort(-(bin_sizes - base))
        for i in spare_order:
            if deficit == 0:
                break
            spare = int(bin_sizes[i] - base[i])
            if spare <= 0:
                continue
            take = min(spare, deficit)
            base[i] += take
            deficit -= take

    return base.astype(np.int64)


def stratified_case_split(y_curve: np.ndarray, test_ratio=0.2, val_ratio=0.1, seed=42):
    rng = np.random.default_rng(seed)
    n_cases = len(y_curve)
    target_test = int(round(n_cases * test_ratio))
    target_val = int(round(n_cases * val_ratio))
    if target_test + target_val >= n_cases:
        raise ValueError("test_ratio + val_ratio must leave at least one training sample")

    anchor = y_curve[:, -1]
    bins = np.where(anchor == 0.0, 0, np.where(anchor == 1.0, 2, 1))
    unique_bins = np.unique(bins)
    grouped_indices = []
    bin_sizes = []

    for b in unique_bins:
        idx = np.where(bins == b)[0]
        rng.shuffle(idx)
        grouped_indices.append(idx)
        bin_sizes.append(len(idx))

    bin_sizes = np.array(bin_sizes, dtype=np.int64)
    test_counts = _allocate_stratified_counts(bin_sizes, target_test)
    remaining_sizes = bin_sizes - test_counts
    val_counts = _allocate_stratified_counts(remaining_sizes, target_val)

    train_parts, val_parts, test_parts = [], [], []
    for idx, n_test, n_val in zip(grouped_indices, test_counts, val_counts):
        test_idx = idx[:n_test]
        val_idx = idx[n_test:n_test + n_val]
        train_idx = idx[n_test + n_val:]
        train_parts.append(train_idx)
        val_parts.append(val_idx)
        test_parts.append(test_idx)

    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    test_idx = np.concatenate(test_parts)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    assert len(train_idx) + len(val_idx) + len(test_idx) == n_cases
    assert len(val_idx) == target_val, (len(val_idx), target_val)
    assert len(test_idx) == target_test, (len(test_idx), target_test)
    assert len(np.intersect1d(train_idx, val_idx)) == 0
    assert len(np.intersect1d(train_idx, test_idx)) == 0
    assert len(np.intersect1d(val_idx, test_idx)) == 0
    return train_idx, val_idx, test_idx


train_idx, val_idx, test_idx = stratified_case_split(
    y_case,
    test_ratio=TEST_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

X_train, X_val, X_test = X_case[train_idx], X_case[val_idx], X_case[test_idx]
y_train, y_val, y_test = y_case[train_idx], y_case[val_idx], y_case[test_idx]

mu = X_train.mean(axis=(0, 1), keepdims=True)
sd = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
X_train = (X_train - mu) / sd
X_val = (X_val - mu) / sd
X_test = (X_test - mu) / sd

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)
print("Split counts (cases):", {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})
print("Ratios:", {"train": round(len(train_idx) / len(y_case), 4), "val": round(len(val_idx) / len(y_case), 4), "test": round(len(test_idx) / len(y_case), 4)})


class SurvivalDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(SurvivalDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SurvivalDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SurvivalDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class ConvBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dropout: float = 0.1):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TemporalBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x) + self.skip(x)


class GRUEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.in_proj(x)
        _, h_n = self.gru(x)
        return h_n[-1]


class CNNEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock1D(input_dim, 64, kernel_size=7, dropout=dropout),
            ConvBlock1D(64, 128, kernel_size=5, dropout=dropout),
            ConvBlock1D(128, hidden_dim, kernel_size=3, dropout=dropout),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        return self.backbone(x).squeeze(-1)


class TCNEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, channels=(64, 128, 128), dropout: float = 0.1):
        super().__init__()
        blocks = []
        prev = input_dim
        for i, ch in enumerate(channels):
            blocks.append(TemporalBlock(prev, ch, kernel_size=3, dilation=2 ** i, dropout=dropout))
            prev = ch
        self.tcn = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(prev, hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        h = self.tcn(x)
        h = self.pool(h).squeeze(-1)
        return self.proj(h)


class Encoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, encoder_type: str = "gru", dropout: float = 0.1):
        super().__init__()
        encoder_type = encoder_type.lower()
        if encoder_type == "gru":
            self.encoder = GRUEncoder(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout)
        elif encoder_type == "cnn":
            self.encoder = CNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout)
        elif encoder_type == "tcn":
            self.encoder = TCNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout)
        else:
            raise ValueError(f"Unsupported encoder_type: {encoder_type}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


class AutoregressiveDecoder(nn.Module):
    def __init__(self, hidden_dim: int, horizon: int, dropout: float = 0.1):
        super().__init__()
        self.horizon = horizon
        self.step_embed = nn.Embedding(horizon, hidden_dim)
        self.input_proj = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gru_cell = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)
        self.hidden_norm = nn.LayerNorm(hidden_dim)
        self.alpha_head = nn.Sequential(
            nn.Linear(2 * hidden_dim + 1, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, context: torch.Tensor, targets: torch.Tensor | None = None, teacher_forcing: bool = False) -> torch.Tensor:
        batch_size = context.size(0)
        hidden = context
        prev_p = torch.ones(batch_size, 1, device=context.device, dtype=context.dtype)
        preds = []

        for step in range(self.horizon):
            step_ids = torch.full((batch_size,), step, device=context.device, dtype=torch.long)
            step_feat = self.step_embed(step_ids)
            decoder_in = self.input_proj(torch.cat([prev_p, step_feat], dim=-1))
            hidden = self.hidden_norm(self.gru_cell(decoder_in, hidden))
            alpha_logits = self.alpha_head(torch.cat([hidden, step_feat, prev_p], dim=-1))
            alpha = torch.sigmoid(alpha_logits)
            p_t = prev_p * alpha
            preds.append(p_t)

            if teacher_forcing and targets is not None:
                prev_p = targets[:, step:step + 1]
            else:
                prev_p = p_t

        return torch.cat(preds, dim=1)


class Seq2SeqSurvivalModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, horizon: int, encoder_type: str = "gru", dropout: float = 0.1):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, hidden_dim=hidden_dim, encoder_type=encoder_type, dropout=dropout)
        self.decoder = AutoregressiveDecoder(hidden_dim=hidden_dim, horizon=horizon, dropout=dropout)

    def forward(self, x: torch.Tensor, targets: torch.Tensor | None = None, teacher_forcing: bool = False) -> torch.Tensor:
        context = self.encoder(x)
        return self.decoder(context=context, targets=targets, teacher_forcing=teacher_forcing)


In [ ]:
class BinomialSurvivalLoss(nn.Module):
    def __init__(
        self,
        n_trials: int = GROUP_SIZE,
        time_decay: float = 0.95,
        eps: float = 1e-6,
        include_combinatorial: bool = True,
    ):
        super().__init__()
        self.n_trials = float(n_trials)
        self.time_decay = time_decay
        self.eps = eps
        self.include_combinatorial = include_combinatorial

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        if y_pred.shape != y_true.shape:
            raise ValueError(f"Shape mismatch: {y_pred.shape} vs {y_true.shape}")

        horizon = y_pred.size(1)
        weights = torch.tensor(
            [self.time_decay ** k for k in range(horizon)],
            device=y_pred.device,
            dtype=y_pred.dtype,
        )
        weights = weights / weights.sum()

        p = y_pred.clamp(self.eps, 1.0 - self.eps)
        n = torch.full_like(y_true, self.n_trials)
        observed_count = torch.round(y_true * self.n_trials).clamp(0.0, self.n_trials)

        log_prob = observed_count * torch.log(p) + (n - observed_count) * torch.log1p(-p)
        if self.include_combinatorial:
            log_prob = log_prob + torch.lgamma(n + 1.0) - torch.lgamma(observed_count + 1.0) - torch.lgamma(n - observed_count + 1.0)

        per_step_nll = -log_prob
        return (per_step_nll * weights.view(1, horizon)).sum(dim=1).mean()


@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()
    losses = []
    preds_all = []
    targets_all = []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        pred = model(xb)
        loss = criterion(pred, yb)
        losses.append(float(loss.item()))
        preds_all.append(pred.detach().cpu())
        targets_all.append(yb.detach().cpu())

    preds = torch.cat(preds_all, dim=0).numpy()
    targets = torch.cat(targets_all, dim=0).numpy()
    mae_per_horizon = np.mean(np.abs(preds - targets), axis=0)
    mse_per_horizon = np.mean((preds - targets) ** 2, axis=0)
    overall_mae = float(np.mean(np.abs(preds - targets)))
    monotone_fraction = float(np.mean(np.all(preds[:, 1:] <= preds[:, :-1] + 1e-6, axis=1)))

    return {
        "loss": float(np.mean(losses)),
        "overall_mae": overall_mae,
        "mae_per_horizon": mae_per_horizon,
        "mse_per_horizon": mse_per_horizon,
        "monotone_fraction": monotone_fraction,
        "preds": preds,
        "targets": targets,
    }


def evaluate_mean_curve_baseline(y_ref: np.ndarray, y_eval: np.ndarray, criterion: nn.Module):
    mean_curve = y_ref.mean(axis=0, keepdims=True).astype(np.float32)
    preds = np.repeat(mean_curve, repeats=len(y_eval), axis=0)
    preds_t = torch.tensor(preds, dtype=torch.float32)
    targets_t = torch.tensor(y_eval, dtype=torch.float32)
    loss = float(criterion(preds_t, targets_t).item())
    mae_per_horizon = np.mean(np.abs(preds - y_eval), axis=0)
    mse_per_horizon = np.mean((preds - y_eval) ** 2, axis=0)
    overall_mae = float(np.mean(np.abs(preds - y_eval)))
    monotone_fraction = float(np.mean(np.all(preds[:, 1:] <= preds[:, :-1] + 1e-6, axis=1)))
    return {
        "loss": loss,
        "overall_mae": overall_mae,
        "mae_per_horizon": mae_per_horizon,
        "mse_per_horizon": mse_per_horizon,
        "monotone_fraction": monotone_fraction,
        "preds": preds,
        "targets": y_eval,
        "mean_curve": mean_curve.squeeze(0),
    }


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    epochs=80,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    teacher_forcing_ratio=0.0,
    scheduler_patience=4,
    scheduler_factor=0.5,
    early_stopping_patience=12,
    early_stopping_min_delta=1e-4,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=scheduler_factor, patience=scheduler_patience)

    history = {"train_loss": [], "val_loss": [], "lr": []}
    best_state = None
    best_val = float("inf")
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            use_tf = teacher_forcing_ratio > 0.0 and random.random() < teacher_forcing_ratio
            pred = model(xb, targets=yb if use_tf else None, teacher_forcing=use_tf)
            loss = criterion(pred, yb)
            loss.backward()
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            total_n += bs

        train_loss = total_loss / max(total_n, 1)
        val_metrics = evaluate_model(model, val_loader, criterion)
        val_loss = val_metrics["loss"]
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        improved = val_loss < (best_val - early_stopping_min_delta)
        if improved:
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"epoch {epoch:03d} | train_loss {train_loss:.6f} | val_loss {val_loss:.6f} | "
            f"val_mae {val_metrics['overall_mae']:.6f} | lr {current_lr:.2e} | patience {patience_counter}/{early_stopping_patience}"
        )

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch:03d}; best epoch was {best_epoch:03d} with val_loss {best_val:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, {"best_epoch": best_epoch, "best_val": best_val}


def print_metrics(name: str, metrics: dict):
    print(f"{name} loss             : {metrics['loss']:.6f}")
    print(f"{name} overall_mae      : {metrics['overall_mae']:.6f}")
    print(f"{name} mae_per_horizon  : {np.round(metrics['mae_per_horizon'], 6)}")
    print(f"{name} mse_per_horizon  : {np.round(metrics['mse_per_horizon'], 6)}")
    print(f"{name} monotone_fraction: {metrics['monotone_fraction']:.4f}")


In [ ]:
criterion = BinomialSurvivalLoss(n_trials=SURVIVAL_TRIALS, time_decay=1.0, include_combinatorial=True)

baseline_val_metrics = evaluate_mean_curve_baseline(y_train, y_val, criterion)
baseline_test_metrics = evaluate_mean_curve_baseline(y_train, y_test, criterion)

print("Mean-curve baseline:")
print_metrics("BASELINE VAL", baseline_val_metrics)
print()
print_metrics("BASELINE TEST", baseline_test_metrics)
print()

model = Seq2SeqSurvivalModel(
    input_dim=NUM_FEATURES,
    hidden_dim=HIDDEN_DIM,
    horizon=len(FUTURE_TIMES),
    encoder_type=ENCODER_TYPE,
    dropout=DROPOUT,
).to(DEVICE)

model, history, fit_info = train_model(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    scheduler_patience=SCHEDULER_PATIENCE,
    scheduler_factor=SCHEDULER_FACTOR,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

print("Best fit info:", fit_info)


In [ ]:
val_metrics = evaluate_model(model, val_loader, criterion)
test_metrics = evaluate_model(model, test_loader, criterion)

print_metrics("MODEL VAL", val_metrics)
print()
print_metrics("MODEL TEST", test_metrics)
print()
print("Improvement over mean-curve baseline on TEST overall_mae:", baseline_test_metrics["overall_mae"] - test_metrics["overall_mae"])

fig, axes = plt.subplots(1, 3, figsize=(11, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss History")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["lr"], color="tab:green")
axes[1].set_title("Learning Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")

example_ids = np.arange(min(5, len(test_metrics["preds"])))
for idx in example_ids:
    axes[2].plot(FUTURE_TIMES, test_metrics["targets"][idx], marker="o", alpha=0.7, color="tab:blue")
    axes[2].plot(FUTURE_TIMES, baseline_test_metrics["preds"][idx], marker="s", alpha=0.35, color="tab:gray")
    axes[2].plot(FUTURE_TIMES, test_metrics["preds"][idx], marker="x", alpha=0.7, color="tab:orange")
axes[2].set_title("Target vs Baseline vs Model")
axes[2].set_xlabel("Future time")
axes[2].set_ylabel("Survival probability")
axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("Mean training survival curve:", np.round(y_train.mean(axis=0), 4))
print("Mean testing  survival curve:", np.round(y_test.mean(axis=0), 4))


## Hierarchical Set-Seq Model

This section keeps the 8 perturbation trajectories explicitly instead of averaging them first.

Pipeline:

1. Input: `(B, 8, T_IN, 15)`
2. Shared trajectory encoder: encode each of the 8 trajectories independently
3. Cross-trajectory attention aggregator: summarize the perturbation set into a case embedding
4. Monotone autoregressive decoder: predict the future survival curve


In [ ]:
def build_case_set_inputs(X: np.ndarray, case_ids: np.ndarray, group_size: int = GROUP_SIZE) -> np.ndarray:
    xs = []
    for case_id in np.unique(case_ids):
        group = X[case_ids == case_id]
        if len(group) != group_size:
            raise ValueError(f"Case {case_id} has {len(group)} trajectories, expected {group_size}")
        xs.append(group)
    return np.stack(xs).astype(np.float32)


X_case_set = build_case_set_inputs(X_traj, case_ids=case_ids, group_size=GROUP_SIZE)
assert X_case_set.shape[:3] == (len(y_case), GROUP_SIZE, T_IN)
assert np.allclose(y_case, build_survival_targets(time_unbound, case_ids=case_ids, future_times=FUTURE_TIMES))

X_set_train = X_case_set[train_idx]
X_set_val = X_case_set[val_idx]
X_set_test = X_case_set[test_idx]

set_mu = X_set_train.mean(axis=(0, 1, 2), keepdims=True)
set_sd = X_set_train.std(axis=(0, 1, 2), keepdims=True) + 1e-6
X_set_train = (X_set_train - set_mu) / set_sd
X_set_val = (X_set_val - set_mu) / set_sd
X_set_test = (X_set_test - set_mu) / set_sd

print("Hierarchical inputs:")
print("  Train:", X_set_train.shape, y_train.shape)
print("  Val  :", X_set_val.shape, y_val.shape)
print("  Test :", X_set_test.shape, y_test.shape)


class SetSurvivalDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


set_train_loader = DataLoader(SetSurvivalDataset(X_set_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
set_val_loader = DataLoader(SetSurvivalDataset(X_set_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
set_test_loader = DataLoader(SetSurvivalDataset(X_set_test, y_test), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
HIER_HIDDEN_DIM = 128
HIER_NUM_HEADS = 4
HIER_DROPOUT = 0.20
HIER_EPOCHS = 3 if LOCAL_SMOKE_TEST else 80


class SharedTrajectoryGRUEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.in_proj(x)
        _, h_n = self.gru(x)
        return h_n[-1]


class CrossTrajectoryAttentionAggregator(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, 2 * hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * hidden_dim, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.summary_proj = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, traj_embeddings: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        attn_out, attn_weights = self.attn(traj_embeddings, traj_embeddings, traj_embeddings)
        x = self.norm1(traj_embeddings + attn_out)
        x = self.norm2(x + self.ffn(x))
        mean_summary = x.mean(dim=1)
        std_summary = x.std(dim=1, unbiased=False)
        case_embedding = self.summary_proj(torch.cat([mean_summary, std_summary], dim=-1))
        return case_embedding, attn_weights


class HierarchicalSetSeqSurvivalModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        horizon: int,
        num_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.traj_encoder = SharedTrajectoryGRUEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )
        self.set_aggregator = CrossTrajectoryAttentionAggregator(
            hidden_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
        )
        self.decoder = AutoregressiveDecoder(
            hidden_dim=hidden_dim,
            horizon=horizon,
            dropout=dropout,
        )

    def forward(self, x: torch.Tensor, targets: torch.Tensor | None = None, teacher_forcing: bool = False) -> torch.Tensor:
        batch_size, group_size, seq_len, feat_dim = x.shape
        x_flat = x.view(batch_size * group_size, seq_len, feat_dim)
        traj_embeddings = self.traj_encoder(x_flat)
        traj_embeddings = traj_embeddings.view(batch_size, group_size, -1)
        case_embedding, _ = self.set_aggregator(traj_embeddings)
        return self.decoder(case_embedding, targets=targets, teacher_forcing=teacher_forcing)


@torch.no_grad()
def inspect_set_attention(model: nn.Module, x: torch.Tensor):
    model.eval()
    batch_size, group_size, seq_len, feat_dim = x.shape
    x_flat = x.view(batch_size * group_size, seq_len, feat_dim)
    traj_embeddings = model.traj_encoder(x_flat).view(batch_size, group_size, -1)
    _, attn_weights = model.set_aggregator(traj_embeddings)
    return attn_weights.detach().cpu().numpy()


In [ ]:
hier_model = HierarchicalSetSeqSurvivalModel(
    input_dim=NUM_FEATURES,
    hidden_dim=HIER_HIDDEN_DIM,
    horizon=len(FUTURE_TIMES),
    num_heads=HIER_NUM_HEADS,
    dropout=HIER_DROPOUT,
).to(DEVICE)

hier_model, hier_history, hier_fit_info = train_model(
    hier_model,
    train_loader=set_train_loader,
    val_loader=set_val_loader,
    criterion=criterion,
    epochs=HIER_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    scheduler_patience=SCHEDULER_PATIENCE,
    scheduler_factor=SCHEDULER_FACTOR,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

hier_val_metrics = evaluate_model(hier_model, set_val_loader, criterion)
hier_test_metrics = evaluate_model(hier_model, set_test_loader, criterion)

print("Hierarchical fit info:", hier_fit_info)
print()
print_metrics("HIER VAL", hier_val_metrics)
print()
print_metrics("HIER TEST", hier_test_metrics)
print()
print("Improvement over mean baseline on TEST overall_mae:", baseline_test_metrics["overall_mae"] - hier_test_metrics["overall_mae"])
print("Improvement over flat case model on TEST overall_mae:", test_metrics["overall_mae"] - hier_test_metrics["overall_mae"])

fig, axes = plt.subplots(1, 3, figsize=(11, 4))

axes[0].plot(hier_history["train_loss"], label="train")
axes[0].plot(hier_history["val_loss"], label="val")
axes[0].set_title("Hierarchical Loss History")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(hier_history["lr"], color="tab:green")
axes[1].set_title("Hierarchical Learning Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")

example_ids = np.arange(min(5, len(hier_test_metrics["preds"])))
for idx in example_ids:
    axes[2].plot(FUTURE_TIMES, hier_test_metrics["targets"][idx], marker="o", alpha=0.7, color="tab:blue")
    axes[2].plot(FUTURE_TIMES, baseline_test_metrics["preds"][idx], marker="s", alpha=0.25, color="tab:gray")
    axes[2].plot(FUTURE_TIMES, test_metrics["preds"][idx], marker="x", alpha=0.5, color="tab:orange")
    axes[2].plot(FUTURE_TIMES, hier_test_metrics["preds"][idx], marker="^", alpha=0.8, color="tab:red")
axes[2].set_title("Target vs Baseline vs Flat vs Hier")
axes[2].set_xlabel("Future time")
axes[2].set_ylabel("Survival probability")
axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

attn_example = torch.tensor(X_set_test[:1], dtype=torch.float32, device=DEVICE)
attn_weights = inspect_set_attention(hier_model, attn_example)
print("Example cross-trajectory attention shape:", attn_weights.shape)


## Temporal + Set Dual Attention Model

This experiment upgrades the trajectory encoder from GRU to temporal self-attention, while keeping the set-level cross-trajectory attention and the monotone autoregressive decoder.

Pipeline:

1. Temporal self-attention inside each trajectory: `(T_IN, 15) -> d`
2. Set attention across the 8 perturbation trajectories: `(8, d) -> d_case`
3. Monotone autoregressive decoder: `d_case -> survival curve`


In [ ]:
DUAL_HIDDEN_DIM = 128
DUAL_NUM_HEADS_TEMPORAL = 4
DUAL_NUM_HEADS_SET = 4
DUAL_NUM_TEMPORAL_LAYERS = 2
DUAL_DROPOUT = 0.20
DUAL_EPOCHS = 3 if LOCAL_SMOKE_TEST else 80


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]


class TemporalSelfAttentionTrajectoryEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
        max_len: int = 256,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.pos_enc = SinusoidalPositionalEncoding(hidden_dim, max_len=max_len + 1)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_norm = nn.LayerNorm(hidden_dim)

        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, T, F)
        x = self.input_proj(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_enc(x)
        x = self.dropout(x)
        x = self.encoder(x)
        return self.out_norm(x[:, 0, :])


class DualAttentionSetSeqSurvivalModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        horizon: int,
        temporal_heads: int = 4,
        temporal_layers: int = 2,
        set_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.traj_encoder = TemporalSelfAttentionTrajectoryEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_heads=temporal_heads,
            num_layers=temporal_layers,
            dropout=dropout,
            max_len=T_IN,
        )
        self.set_aggregator = CrossTrajectoryAttentionAggregator(
            hidden_dim=hidden_dim,
            num_heads=set_heads,
            dropout=dropout,
        )
        self.decoder = AutoregressiveDecoder(
            hidden_dim=hidden_dim,
            horizon=horizon,
            dropout=dropout,
        )

    def forward(self, x: torch.Tensor, targets: torch.Tensor | None = None, teacher_forcing: bool = False) -> torch.Tensor:
        batch_size, group_size, seq_len, feat_dim = x.shape
        x_flat = x.view(batch_size * group_size, seq_len, feat_dim)
        traj_embeddings = self.traj_encoder(x_flat).view(batch_size, group_size, -1)
        case_embedding, _ = self.set_aggregator(traj_embeddings)
        return self.decoder(case_embedding, targets=targets, teacher_forcing=teacher_forcing)


In [ ]:
dual_model = DualAttentionSetSeqSurvivalModel(
    input_dim=NUM_FEATURES,
    hidden_dim=DUAL_HIDDEN_DIM,
    horizon=len(FUTURE_TIMES),
    temporal_heads=DUAL_NUM_HEADS_TEMPORAL,
    temporal_layers=DUAL_NUM_TEMPORAL_LAYERS,
    set_heads=DUAL_NUM_HEADS_SET,
    dropout=DUAL_DROPOUT,
).to(DEVICE)

dual_model, dual_history, dual_fit_info = train_model(
    dual_model,
    train_loader=set_train_loader,
    val_loader=set_val_loader,
    criterion=criterion,
    epochs=DUAL_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    grad_clip_norm=GRAD_CLIP_NORM,
    teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    scheduler_patience=SCHEDULER_PATIENCE,
    scheduler_factor=SCHEDULER_FACTOR,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

dual_val_metrics = evaluate_model(dual_model, set_val_loader, criterion)
dual_test_metrics = evaluate_model(dual_model, set_test_loader, criterion)

print("Dual-attention fit info:", dual_fit_info)
print()
print_metrics("DUAL VAL", dual_val_metrics)
print()
print_metrics("DUAL TEST", dual_test_metrics)
print()
print("Improvement over mean baseline on TEST overall_mae:", baseline_test_metrics["overall_mae"] - dual_test_metrics["overall_mae"])
print("Improvement over flat case model on TEST overall_mae:", test_metrics["overall_mae"] - dual_test_metrics["overall_mae"])
print("Improvement over hierarchical set-seq model on TEST overall_mae:", hier_test_metrics["overall_mae"] - dual_test_metrics["overall_mae"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(dual_history["train_loss"], label="train")
axes[0].plot(dual_history["val_loss"], label="val")
axes[0].set_title("Dual-Attention Loss History")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(dual_history["lr"], color="tab:green")
axes[1].set_title("Dual-Attention Learning Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")

plt.tight_layout()
plt.show()

example_ids = np.arange(min(5, len(dual_test_metrics["preds"])))
fig, example_axes = plt.subplots(len(example_ids), 1, figsize=(8, 2.8 * len(example_ids)), sharex=True)
if len(example_ids) == 1:
    example_axes = [example_axes]

model_styles = [
    ("Target", dual_test_metrics["targets"], "tab:blue", "o", "-", 2.4),
    ("Mean Baseline", baseline_test_metrics["preds"], "tab:gray", "s", "--", 1.8),
    ("Flat GRU Seq2Seq", test_metrics["preds"], "tab:orange", "x", "-.", 1.8),
    ("Hierarchical Set-Seq", hier_test_metrics["preds"], "tab:red", "^", "-", 1.8),
    ("Dual Attention", dual_test_metrics["preds"], "tab:purple", "D", "-", 2.0),
]

for ax, idx in zip(example_axes, example_ids):
    for label, values, color, marker, linestyle, linewidth in model_styles:
        ax.plot(
            FUTURE_TIMES,
            values[idx],
            label=label,
            color=color,
            marker=marker,
            linestyle=linestyle,
            linewidth=linewidth,
            markersize=6,
            alpha=0.95,
        )
    ax.set_title(f"Example case {idx}")
    ax.set_ylabel("Survival")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True)

example_axes[-1].set_xlabel("Future time")
fig.suptitle("Per-Case Survival Curves: Target vs Model Predictions", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Comprehensive diagnostics after all candidate models have been evaluated.
comparison_metrics = {
    "Mean baseline": baseline_test_metrics,
    "Flat GRU": test_metrics,
    "Hierarchical": hier_test_metrics,
    "Dual attention": dual_test_metrics,
}
comparison_histories = {
    "Flat GRU": history,
    "Hierarchical": hier_history,
    "Dual attention": dual_history,
}
comparison_colors = {
    "Mean baseline": "tab:gray",
    "Flat GRU": "tab:orange",
    "Hierarchical": "tab:red",
    "Dual attention": "tab:purple",
}

def add_curve_metrics(metrics: dict) -> dict:
    preds = metrics["preds"]
    targets = metrics["targets"]
    abs_err = np.abs(preds - targets)
    sq_err = (preds - targets) ** 2
    n_horizon = preds.shape[1]
    thirds = np.array_split(np.arange(n_horizon), 3)
    return {
        "loss": metrics["loss"],
        "overall_mae": float(abs_err.mean()),
        "overall_rmse": float(np.sqrt(sq_err.mean())),
        "final_mae": float(abs_err[:, -1].mean()),
        "early_mae": float(abs_err[:, thirds[0]].mean()),
        "mid_mae": float(abs_err[:, thirds[1]].mean()),
        "late_mae": float(abs_err[:, thirds[2]].mean()),
        "mean_bias": float((preds - targets).mean()),
        "auc_mae": float(np.mean(np.abs(np.trapezoid(preds, FUTURE_TIMES, axis=1) - np.trapezoid(targets, FUTURE_TIMES, axis=1))) / (FUTURE_TIMES[-1] - FUTURE_TIMES[0])),
        "monotone_fraction": metrics["monotone_fraction"],
    }

summary_df = pd.DataFrame({name: add_curve_metrics(metrics) for name, metrics in comparison_metrics.items()}).T
print("Comprehensive test metrics:")
print(summary_df.round(4).to_string())

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metric_panels = [
    ("overall_mae", "Overall MAE"),
    ("overall_rmse", "Overall RMSE"),
    ("final_mae", f"Final-horizon MAE (t={int(FUTURE_TIMES[-1])})"),
    ("early_mae", "Early-window MAE"),
    ("late_mae", "Late-window MAE"),
    ("auc_mae", "Normalized AUC MAE"),
]
for ax, (metric_name, title) in zip(axes.flat, metric_panels):
    values = summary_df[metric_name]
    colors = [comparison_colors[name] for name in values.index]
    ax.bar(values.index, values.values, color=colors, alpha=0.85)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(True, axis="y", alpha=0.25, linestyle=":")
    for i, value in enumerate(values.values):
        ax.text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for name, hist in comparison_histories.items():
    epochs = np.arange(1, len(hist["train_loss"]) + 1)
    axes[0, 0].plot(epochs, hist["train_loss"], label=f"{name} train", color=comparison_colors[name], linestyle="-")
    axes[0, 0].plot(epochs, hist["val_loss"], label=f"{name} val", color=comparison_colors[name], linestyle="--")
axes[0, 0].set_title("Training and Validation Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Binomial NLL")
axes[0, 0].grid(True, alpha=0.25, linestyle=":")
axes[0, 0].legend(fontsize=8, ncol=2)

for name, metrics in comparison_metrics.items():
    axes[0, 1].plot(FUTURE_TIMES, metrics["mae_per_horizon"], label=name, color=comparison_colors[name], linewidth=2)
axes[0, 1].set_title("MAE by Future Horizon")
axes[0, 1].set_xlabel("Future time_unbound threshold")
axes[0, 1].set_ylabel("MAE")
axes[0, 1].grid(True, alpha=0.25, linestyle=":")
axes[0, 1].legend()

axes[1, 0].plot(FUTURE_TIMES, y_test.mean(axis=0), label="Target", color="tab:blue", linewidth=3)
for name, metrics in comparison_metrics.items():
    axes[1, 0].plot(FUTURE_TIMES, metrics["preds"].mean(axis=0), label=name, color=comparison_colors[name], linewidth=2, linestyle="--" if name == "Mean baseline" else "-")
axes[1, 0].set_title("Mean Survival Curve")
axes[1, 0].set_xlabel("Future time_unbound threshold")
axes[1, 0].set_ylabel("Survival probability")
axes[1, 0].set_ylim(-0.05, 1.05)
axes[1, 0].grid(True, alpha=0.25, linestyle=":")
axes[1, 0].legend()

for name, metrics in comparison_metrics.items():
    residual = metrics["preds"] - metrics["targets"]
    axes[1, 1].plot(FUTURE_TIMES, residual.mean(axis=0), label=name, color=comparison_colors[name], linewidth=2)
axes[1, 1].axhline(0.0, color="black", linewidth=1)
axes[1, 1].set_title("Mean Prediction Bias by Horizon")
axes[1, 1].set_xlabel("Future time_unbound threshold")
axes[1, 1].set_ylabel("Prediction - target")
axes[1, 1].grid(True, alpha=0.25, linestyle=":")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

case_mae_by_model = {
    name: np.mean(np.abs(metrics["preds"] - metrics["targets"]), axis=1)
    for name, metrics in comparison_metrics.items()
}
max_case_mae = max(float(values.max()) for values in case_mae_by_model.values())
hist_bins = np.linspace(0.0, max(max_case_mae + 1e-6, 0.05), 25)
for name, case_mae in case_mae_by_model.items():
    axes[0].hist(case_mae, bins=hist_bins, alpha=0.35, label=name, color=comparison_colors[name])
axes[0].set_title("Case-wise MAE Distribution")
axes[0].set_xlabel("MAE per case")
axes[0].set_ylabel("Number of test cases")
axes[0].grid(True, axis="y", alpha=0.25, linestyle=":")
axes[0].legend()

def calibration_table(preds: np.ndarray, targets: np.ndarray, bins: np.ndarray) -> pd.DataFrame:
    flat_pred = preds.reshape(-1)
    flat_target = targets.reshape(-1)
    bin_ids = np.digitize(flat_pred, bins) - 1
    rows = []
    for bin_id in range(len(bins) - 1):
        mask = bin_ids == bin_id
        if mask.sum() == 0:
            continue
        rows.append({
            "bin_left": bins[bin_id],
            "bin_right": bins[bin_id + 1],
            "pred_mean": float(flat_pred[mask].mean()),
            "target_mean": float(flat_target[mask].mean()),
            "count": int(mask.sum()),
        })
    return pd.DataFrame(rows)

calibration_bins = np.linspace(0.0, 1.0, 11)
axes[1].plot([0, 1], [0, 1], color="black", linewidth=1, linestyle=":", label="Ideal")
for name, metrics in comparison_metrics.items():
    cal = calibration_table(metrics["preds"], metrics["targets"], calibration_bins)
    axes[1].plot(cal["pred_mean"], cal["target_mean"], marker="o", label=name, color=comparison_colors[name], linewidth=2)
axes[1].set_title("Calibration Across Case-Horizon Pairs")
axes[1].set_xlabel("Mean predicted survival")
axes[1].set_ylabel("Mean observed survival fraction")
axes[1].set_xlim(-0.03, 1.03)
axes[1].set_ylim(-0.03, 1.03)
axes[1].grid(True, alpha=0.25, linestyle=":")
axes[1].legend()

plt.tight_layout()
plt.show()

best_model_name = summary_df["overall_mae"].idxmin()
best_metrics = comparison_metrics[best_model_name]
ordered_case_ids = np.argsort(np.mean(np.abs(best_metrics["preds"] - best_metrics["targets"]), axis=1))
case_ids_to_plot = [ordered_case_ids[0], ordered_case_ids[len(ordered_case_ids) // 2], ordered_case_ids[-1]]
case_titles = ["Best case", "Median-error case", "Worst case"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
for ax, case_id, title in zip(axes, case_ids_to_plot, case_titles):
    ax.plot(FUTURE_TIMES, best_metrics["targets"][case_id], label="Target", color="tab:blue", linewidth=3)
    for name, metrics in comparison_metrics.items():
        ax.plot(FUTURE_TIMES, metrics["preds"][case_id], label=name, color=comparison_colors[name], linewidth=2, linestyle="--" if name == "Mean baseline" else "-")
    ax.set_title(f"{title}: test index {case_id} | best={best_model_name}")
    ax.set_xlabel("Future time_unbound threshold")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25, linestyle=":")
axes[0].set_ylabel("Survival probability")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()
